In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pandas as pd
import re

data_dir = Path('/content/battery_data/SL_Dataset_SECL_INR21700-M50T')
NOMINAL_CAPACITY = 4.85

In [ ]:
def read_channel_sheet(xlsx_path):
    xl = pd.ExcelFile(xlsx_path)
    channel_sheets = [s for s in xl.sheet_names if s.startswith('Channel_')]
    if not channel_sheets:
        return None
    return pd.read_excel(xlsx_path, sheet_name=channel_sheets[0])


def parse_cell_id(path):
    text = str(path).lower()
    match = re.search(r'_(g1|v4|v5|w8|w9|w10)_', text)
    return match.group(1).upper() if match else None


def parse_rpt_id(path):
    text = str(path)
    match = re.search(r'RPT_(\d+)', text)
    return int(match.group(1)) if match else None

In [ ]:
diagnostic_files = list((data_dir / 'diagnostic_tests').rglob('*.xlsx'))

capacity_files = [
    f for f in diagnostic_files
    if 'Capacity_test_with_pulses' in str(f) and not f.name.startswith('_')
]

print('capacity files:', len(capacity_files))

In [ ]:
label_rows = []
skipped = []

for i, file in enumerate(capacity_files):
    if i % 10 == 0:
        print(i, '/', len(capacity_files), file.name)

    cell_id = parse_cell_id(file)
    rpt_id = parse_rpt_id(file)

    if cell_id is None or rpt_id is None:
        skipped.append((file, 'parse failed'))
        continue

    try:
        df = read_channel_sheet(file)
        if df is None:
            skipped.append((file, 'no channel sheet'))
            continue

        discharge_capacity = df['Discharge_Capacity(Ah)'].max()
        soh = discharge_capacity / NOMINAL_CAPACITY

        label_rows.append({
            'cell_id': cell_id,
            'rpt_id': rpt_id,
            'measured_capacity_ah': discharge_capacity,
            'soh': soh,
            'file': str(file)
        })

    except Exception as e:
        skipped.append((file, str(e)))

labels = pd.DataFrame(label_rows)

In [ ]:
labels_clean = (
    labels
    .groupby(['cell_id', 'rpt_id'], as_index=False)
    .agg({
        'measured_capacity_ah': 'mean',
        'soh': 'mean',
        'file': 'first'
    })
    .sort_values(['cell_id', 'rpt_id'])
    .reset_index(drop=True)
)

print(labels_clean.shape)
display(labels_clean.groupby('cell_id')['soh'].describe())
display(labels_clean.pivot_table(index='rpt_id', columns='cell_id', values='soh'))

In [ ]:
out_path = '/content/drive/MyDrive/battery_data/stanford_soh_labels.csv'
labels_clean.to_csv(out_path, index=False)

print(out_path)